# 09 Post-evaluation diagnostics

This notebook adds two checks after the main evaluation is complete. They look more closely at two limits of the evaluation evidence: whether category errors may differ over time, and how difficult the recommender relevance test is. Nothing here changes a model, a rule, a label, an approval or any result in the other notebooks.

| Field | Value |
|---|---|
| **Purpose** | Describe two limits of the evaluation evidence after the evaluation is scored |
| **Assignment requirement** | Supports the evaluation of the job categories (Tasks 2 and 3) and of the recommender (Task 5) |
| **Inputs** | `label_evaluation_items.csv`, `jobs_clean.parquet`, `recommend_relevance_items.csv`, `recommend_relevance_summary.csv` |
| **Outputs** | None |
| **Build identity** | Printed by the first code cell |

## Status of these checks

- I add these checks after the category and recommender evaluations are scored. They are descriptive diagnostics, not part of model selection.
- The categoriser, the recommender, the labels and the approval record stay exactly as evaluated. No result in this notebook is used to choose or change a method.

In [1]:
from job_market.notebook import start

project = start()

RUN_ID: 20261011T132714Z-e537465460
Source: data/raw/all_upwork_jobs_2024-02-07-2024-03-24.csv (244,828 rows)
Taxonomy version: 1.0
Verified outputs: 54 file(s), all hashes match
Environment: matches environment.yml


In [2]:
import numpy as np
import pandas as pd
from scipy import stats

from job_market.evaluate import wilson_interval
from job_market.taxonomy import CATEGORIES

pd.set_option("display.width", 170)

items = project.read_csv("label_evaluation_items.csv")
jobs = project.read_parquet("jobs_clean.parquet", columns=["row_id", "posted_date", "in_timeseries_window"])
relevance = project.read_csv("recommend_relevance_items.csv")
summary = project.read_csv("recommend_relevance_summary.csv")
name = {category: label for category, (label, _) in CATEGORIES.items()}

## 1. Category agreement over time

The blind set is a random sample of 350 jobs, drawn before any rule tuning. Each title has a human label and a system prediction. The posting dates spread over the whole collection period, so the sample can be split by date.

I split the 37-day window at 5 March: 16 February to 4 March (18 days) and 5 to 23 March (19 days). Blind titles posted outside the window are shown separately, because no time-based analysis uses those days.

In [3]:
EARLY, LATE, OUTSIDE = "16 Feb to 4 Mar", "5 to 23 Mar", "outside the window"
blind = items[items["label_set"] == "blind"].merge(jobs, on="row_id", how="left", validate="one_to_one")
assert blind["posted_date"].notna().all()
posted = pd.to_datetime(blind["posted_date"])
blind["period"] = np.select([~blind["in_timeseries_window"], posted < pd.Timestamp("2024-03-05")],
                            [OUTSIDE, EARLY], default=LATE)
blind["period"].value_counts().reindex([EARLY, LATE, OUTSIDE])

period
16 Feb to 4 Mar       168
5 to 23 Mar           163
outside the window     19
Name: count, dtype: int64

### 1.1 Agreement in each period

Agreement is the share of titles where the system's primary category equals the human label. The interval is the 95% Wilson interval, the same interval the main evaluation uses.

In [4]:
def agreement_row(group: pd.DataFrame) -> dict:
    correct, titles = int(group["correct"].sum()), len(group)
    low, high = wilson_interval(correct, titles)
    return {"titles": titles, "correct": correct, "agreement %": round(100 * correct / titles, 1),
            "95% low %": round(100 * low, 1), "95% high %": round(100 * high, 1)}

agreement = pd.DataFrame.from_dict({period: agreement_row(blind[blind["period"] == period])
                                    for period in [EARLY, LATE, OUTSIDE]}, orient="index")
agreement

,titles,correct,agreement %,95% low %,95% high %
16 Feb to 4 Mar,168,117,69.6,62.3,76.1
5 to 23 Mar,163,106,65.0,57.4,71.9
outside the window,19,18,94.7,75.4,99.1


### 1.2 Difference between the two periods

The difference in agreement gets a 95% interval from Newcombe's method, which combines the two Wilson intervals. A chi-square test of the two-by-two table of correct and incorrect titles gives the p-value.

In [5]:
(k1, n1), (k2, n2) = [(int(blind.loc[blind["period"] == p, "correct"].sum()), int((blind["period"] == p).sum()))
                      for p in (EARLY, LATE)]
p1, p2 = k1 / n1, k2 / n2
(l1, u1), (l2, u2) = wilson_interval(k1, n1), wilson_interval(k2, n2)
difference = p1 - p2
diff_low = difference - np.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2)
diff_high = difference + np.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2)
p_value = stats.chi2_contingency([[k1, n1 - k1], [k2, n2 - k2]], correction=False)[1]
pd.Series({"difference (points)": round(100 * difference, 1), "95% low (points)": round(100 * diff_low, 1),
           "95% high (points)": round(100 * diff_high, 1), "p-value": round(p_value, 2)})

difference (points)     4.60
95% low (points)       -5.50
95% high (points)      14.60
p-value                 0.37
dtype: float64

- Agreement is 69.6% (117 of 168 titles) in the first period and 65.0% (106 of 163) in the second, a difference of 4.6 percentage points.
- The 95% interval for the difference runs from −5.5 to +14.6 points, and the test gives p = 0.37. The observed agreement differs between the two periods, but this sample is too small to establish whether classification reliability changes over time. The result does not show that the error rate stays the same either.
- The 19 titles outside the window (18 correct) are too few to interpret.

### 1.3 Human labels per category in each period

Overall agreement can stay similar while the types of error change. The table shows how many blind titles each category has in each period, which limits what can be said about any single category.

In [6]:
window = blind[blind["period"] != OUTSIDE]
label_counts = pd.crosstab(window["human"].map(name), window["period"]).reindex(columns=[EARLY, LATE])
label_counts["total"] = label_counts.sum(axis=1)
label_counts.sort_values("total", ascending=False)

period,16 Feb to 4 Mar,5 to 23 Mar,total
human,,,
Design & Creative,20,24,44
Software & IT,26,13,39
Digital Marketing,14,23,37
Web Development,16,19,35
Other / Unclear,14,8,22
Video & Animation,13,6,19
Writing & Content,8,10,18
Sales & Lead Generation,8,8,16
AI & Machine Learning,8,4,12


- Only four categories have at least 10 human labels in both periods: Software & IT, Design & Creative, Digital Marketing and Web Development.
- The mix of categories differs between the periods. For example, Software & IT has 26 titles in the first period and 13 in the second. Agreement differs between categories, so a change in mix alone can move the overall agreement.
- I do not calculate agreement rates or tests for single categories: most categories have fewer than 10 titles in a period, and many tests on small groups would add more uncertainty than they remove.

### 1.4 Other / Unclear in each period

These are two different comparisons: how often the system predicts Other / Unclear, and how often the human label is Other / Unclear. The denominator is the number of blind titles in the period. They are shares of the blind sample, not of all 229,520 postings in the window; the full-window share of Other / Unclear is monitored in the Task 2 analysis (`trend_other_monitor.csv`).

In [7]:
other = pd.DataFrame({period: {
    "titles": int((group := window[window["period"] == period]).shape[0]),
    "system predicts Other / Unclear": int((group["predicted"] == "other_unclear").sum()),
    "human label Other / Unclear": int((group["human"] == "other_unclear").sum()),
} for period in [EARLY, LATE]}).T
other["system %"] = (100 * other["system predicts Other / Unclear"] / other["titles"]).round(1)
other["human %"] = (100 * other["human label Other / Unclear"] / other["titles"]).round(1)
other

,titles,system predicts Other / Unclear,human label Other / Unclear,system %,human %
16 Feb to 4 Mar,168,24,14,14.3,8.3
5 to 23 Mar,163,23,8,14.1,4.9


- The system predicts Other / Unclear for about 14% of blind titles in both periods (24 of 168 and 23 of 163).
- The human label is Other / Unclear less often: 14 and 8 titles. In both periods the system puts more titles into Other / Unclear than the human does, which matches the low precision of that group in the main evaluation.

### 1.5 Most common disagreements

The table lists pairs of human label and system prediction that differ, when the pair occurs at least 3 times in at least one period. Pairs below this threshold are not shown; that does not mean those pairs have no errors.

In [8]:
wrong = window[window["human"] != window["predicted"]]
pairs = pd.crosstab([wrong["human"].map(name), wrong["predicted"].map(name)], wrong["period"]).reindex(
    columns=[EARLY, LATE], fill_value=0)
pairs.index.names = ["human label", "system prediction"]
pairs["total"] = pairs.sum(axis=1)
common_pairs = pairs[pairs[[EARLY, LATE]].max(axis=1) >= 3].sort_values("total", ascending=False)
common_pairs

period                                                                16 Feb to 4 Mar  5 to 23 Mar  total
human label                system prediction                                                             
Software & IT              Web Development                                          5            3      8
                           Other / Unclear                                          4            2      6
Engineering & Architecture Other / Unclear                                          3            2      5
Admin & Virtual Assistance Other / Unclear                                          3            1      4
Software & IT              Business, Project Management & Consulting                0            3      3
                           Data & Analytics                                         3            0      3
Writing & Content          Other / Unclear                                          0            3      3

- The most common disagreements are Software & IT titles predicted as Web Development (5 and 3) or as Other / Unclear (4 and 2), and Engineering & Architecture titles predicted as Other / Unclear (3 and 2).
- The counts in each period are small, so they show which errors occur, not whether they change over time.

### 1.6 What this means for the category results

- The blind labels cannot establish whether category errors change over time: the difference in overall agreement is uncertain, and most categories have too few labels in each period.
- The Task 2 trend results therefore keep the limit stated with them: they carry the uncertainty of the categories, including any change in the types of error over the window.

## 2. How difficult the recommender relevance test is

The recommender evaluation pools the top 5 results of three methods for 12 freelancer profiles: 12 × 3 × 5 = 180 method-specific slots. When methods return the same title for a profile, that title is judged once, so the pool contains 122 distinct titles. Each is labelled blind to the method: 2 relevant, 1 partly relevant, 0 not relevant.

The candidate pool comes from the three methods being evaluated, not from a random sample of all postings. Each method already returns titles that match the profile's skills, so most pooled titles are likely to be relevant.

### 2.1 Relevance labels over the 122 distinct titles

In [9]:
distinct = relevance.drop_duplicates(["profile_id", "title_key"])
assert len(distinct) == relevance["title_key"].nunique()
label_mix = distinct["relevance"].value_counts().reindex([2, 1, 0]).rename("distinct titles").to_frame()
label_mix["share %"] = (100 * label_mix["distinct titles"] / len(distinct)).round(1)
label_mix

,distinct titles,share %
relevance,,
2,115,94.3
1,6,4.9
0,1,0.8


### 2.2 Relevance labels over each method's 60 slots

Here the denominator is each method's 60 slots (12 profiles × 5 results), not the 122 distinct titles. The share of slots with label 2 is the strict Precision@5, and it matches the main evaluation.

In [10]:
slots = pd.crosstab(relevance["method"], relevance["relevance"]).reindex(columns=[2, 1, 0], fill_value=0)
slots["slots"] = slots.sum(axis=1)
slots["strict Precision@5"] = (slots[2] / slots["slots"]).round(3)
slots["main evaluation"] = summary.set_index("method")["mean_precision"].round(3)
slots

relevance,2,1,0,slots,strict Precision@5,main evaluation
method,,,,,,
keyword,56,3,1,60,0.933,0.933
tfidf_hybrid,58,2,0,60,0.967,0.967
tfidf_word,57,3,0,60,0.950,0.950


- 115 of the 122 distinct titles (94.3%) receive the top label; 6 are partly relevant and 1 is not relevant.
- Per method, 56 (keyword), 57 (word TF-IDF) and 58 (hybrid TF-IDF) of 60 slots hold a fully relevant title. The slot-based strict Precision@5 equals the value in the main evaluation for every method.

### 2.3 What this means for the recommender scores

- When nearly every pooled title is relevant, all three methods score close to the maximum. The high nDCG@5 (0.985) and strict Precision@5 (0.967) show that the selected method returns relevant titles for these 12 profiles.
- They cannot show which method is better in general, or how the methods perform on harder searches with fewer relevant candidates. The selected hybrid method returns 58 fully relevant results in its 60 evaluated slots, compared with 57 for word TF-IDF. This one-result advantage is too small to establish that the hybrid method performs better in general.

## Claims check

In [11]:
claims = {
    "blind set: 168 + 163 + 19 = 350 titles": agreement["titles"].tolist() == [168, 163, 19],
    "first period: 117 of 168 correct (69.6%)": (k1, n1) == (117, 168) and round(100 * p1, 1) == 69.6,
    "second period: 106 of 163 correct (65.0%)": (k2, n2) == (106, 163) and round(100 * p2, 1) == 65.0,
    "difference 4.6 points, interval -5.5 to +14.6": [round(100 * x, 1) for x in (difference, diff_low, diff_high)] == [4.6, -5.5, 14.6],
    "interval includes zero and p = 0.37": diff_low < 0 < diff_high and round(p_value, 2) == 0.37,
    "outside the window: 18 of 19 correct": agreement.loc[OUTSIDE, ["correct", "titles"]].tolist() == [18, 19],
    "four categories with at least 10 labels in both periods": sorted(label_counts.index[(label_counts[[EARLY, LATE]] >= 10).all(axis=1)])
        == ["Design & Creative", "Digital Marketing", "Software & IT", "Web Development"],
    "Software & IT: 26 then 13 titles": label_counts.loc["Software & IT", [EARLY, LATE]].tolist() == [26, 13],
    "system Other / Unclear: 24 and 23": other["system predicts Other / Unclear"].tolist() == [24, 23],
    "human Other / Unclear: 14 and 8": other["human label Other / Unclear"].tolist() == [14, 8],
    "Software & IT as Web Development: 5 and 3": common_pairs.loc[("Software & IT", "Web Development"), [EARLY, LATE]].tolist() == [5, 3],
    "122 distinct titles: 115, 6 and 1": len(distinct) == 122 and label_mix["distinct titles"].tolist() == [115, 6, 1],
    "180 slots, 60 per method": len(relevance) == 180 and (slots["slots"] == 60).all(),
    "fully relevant slots: 56, 57, 58": slots.loc[["keyword", "tfidf_word", "tfidf_hybrid"], 2].tolist() == [56, 57, 58],
    "slot-based Precision@5 matches the main evaluation": (slots["strict Precision@5"] == slots["main evaluation"]).all(),
}
claims = pd.Series(claims, name="true").to_frame()
assert claims["true"].all(), claims[~claims["true"]]
claims

,true
blind set: 168 + 163 + 19 = 350 titles,True
first period: 117 of 168 correct (69.6%),True
second period: 106 of 163 correct (65.0%),True
"difference 4.6 points, interval -5.5 to +14.6",True
interval includes zero and p = 0.37,True
outside the window: 18 of 19 correct,True
four categories with at least 10 labels in both periods,True
Software & IT: 26 then 13 titles,True
system Other / Unclear: 24 and 23,True
human Other / Unclear: 14 and 8,True
